In [ ]:
from google.colab import files
files.upload()

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(
    "/content/Oily-Dry-Skin-Types/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "/content/Oily-Dry-Skin-Types/valid",
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print(train_dataset.class_to_idx)

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

model = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {running_loss:.4f}, "
        f"Train Accuracy: {train_acc:.2f}%"
    )

In [ ]:
torch.save(
    model.state_dict(),
    "skin_classifier.pth"
)

print("Model saved!")
print(model.classifier)

In [ ]:
state_dict = torch.load("skin_classifier.pth", map_location="cpu")

print(state_dict["classifier.1.weight"].shape)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

new_model = efficientnet_b0(weights=None)

new_model.classifier[1] = nn.Linear(
    new_model.classifier[1].in_features,
    3
)

new_model.load_state_dict(
    torch.load("skin_classifier.pth", map_location="cpu")
)

print("Loaded successfully!")

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
from PIL import Image

image = Image.open("/content/best-ingredients-for-oily-skin-01-pg-full.jpg").convert("RGB")

image = transform(image)

image = image.unsqueeze(0)   # Shape: [1, 3, 224, 224]

In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    outputs = model(image)

    probabilities = F.softmax(outputs, dim=1)

    _, predicted = torch.max(outputs, 1)